# 3_Oro



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'dod7dumu05u6cn'
os.environ['DataZoneDomainId'] = 'dzd-crnhdsp0s7155z'
os.environ['DataZoneEnvironmentId'] = 'deeuj7l8vp1nzb'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "dod7dumu05u6cn",
                "DataZoneDomainId": "dzd-crnhdsp0s7155z",
                "DataZoneEnvironmentId": "deeuj7l8vp1nzb",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import pandas as pd
import numpy as np
import boto3
import json
import awswrangler as wr
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Librerías importadas")

✓ Librerías importadas


In [0]:
silver_path = "s3://sin-exam-silver-877617909831/"
gold_path = "s3://sin-exam-gold-877617909831/"
database = "sin-exam_db"

logger.info(f"Configuración cargada - Silver: {silver_path}, Gold: {gold_path}")

# Leer tabla de hechos desde Silver
fact_path = f"{silver_path}fact_ventas/"
logger.info(f"Leyendo fact table desde: {fact_path}")

fact_table = wr.s3.read_parquet(path=fact_path)
logger.info(f"✓ Fact table cargada: {len(fact_table):,} filas")

# Leer dimensiones
dimensions = {}
dim_names = ['dim_tiempo', 'dim_cliente', 'dim_producto', 'dim_ubicacion', 'dim_envio']

for dim_name in dim_names:
    try:
        dim_path = f"{silver_path}{dim_name}/"
        dimensions[dim_name] = wr.s3.read_parquet(path=dim_path)
        logger.info(f"✓ {dim_name} cargada: {len(dimensions[dim_name])} registros")
    except Exception as e:
        logger.warning(f"⚠️  No se pudo cargar {dim_name}: {e}")

print(f"\n✓ Todas las dimensiones y fact table cargadas exitosamente")




✓ Todas las dimensiones y fact table cargadas exitosamente


In [0]:
print("\n" + "="*80)
print("DEFINICIÓN DE KPIS PARA SUPERSTORE")
print("="*80)

kpi_definitions = {
    'kpi_1_ventas_temporales': {
        'descripcion': 'Métricas de ventas y rentabilidad por periodo (mensual/anual)',
        'metricas': ['ventas_totales', 'ganancia_total', 'cantidad_ordenes', 'ticket_promedio', 'margen_%'],
        'dimensiones': ['año', 'mes', 'trimestre'],
        'uso': 'Tendencias y forecasting'
    },
    'kpi_2_top_productos': {
        'descripcion': 'Top productos y categorías más rentables',
        'metricas': ['ventas', 'ganancia', 'cantidad_vendida', 'num_ordenes'],
        'dimensiones': ['producto', 'categoria', 'subcategoria'],
        'uso': 'Análisis de portafolio de productos'
    },
    'kpi_3_analisis_clientes': {
        'descripcion': 'Segmentación y comportamiento de clientes',
        'metricas': ['ventas_por_cliente', 'ganancia_por_cliente', 'frecuencia_compra'],
        'dimensiones': ['segmento', 'region'],
        'uso': 'Customer analytics y targeting'
    },
    'kpi_4_performance_geografico': {
        'descripcion': 'Desempeño por región y estado',
        'metricas': ['ventas', 'ganancia', 'margen', 'ordenes'],
        'dimensiones': ['region', 'estado', 'ciudad'],
        'uso': 'Expansión geográfica y estrategia regional'
    },
    'kpi_5_eficiencia_operacional': {
        'descripcion': 'Métricas operacionales y logística',
        'metricas': ['ventas_por_modo_envio', 'tiempo_entrega_promedio', 'descuentos_aplicados'],
        'dimensiones': ['modo_envio'],
        'uso': 'Optimización de operaciones'
    },
    'kpi_6_metricas_globales': {
        'descripcion': 'KPIs ejecutivos clave (para KPI cards)',
        'metricas': ['ventas_totales', 'ganancia_total', 'num_ordenes', 'num_clientes', 'ticket_promedio', 'margen_promedio'],
        'dimensiones': [],
        'uso': 'Dashboard ejecutivo'
    }
}

for kpi_name, definition in kpi_definitions.items():
    print(f"\n📊 {kpi_name.upper()}:")
    print(f"   Descripción: {definition['descripcion']}")
    print(f"   Métricas: {', '.join(definition['metricas'])}")
    print(f"   Dimensiones: {', '.join(definition['dimensiones']) if definition['dimensiones'] else 'Ninguna'}")
    print(f"   Uso: {definition['uso']}")




DEFINICIÓN DE KPIS PARA SUPERSTORE

📊 KPI_1_VENTAS_TEMPORALES:
   Descripción: Métricas de ventas y rentabilidad por periodo (mensual/anual)
   Métricas: ventas_totales, ganancia_total, cantidad_ordenes, ticket_promedio, margen_%
   Dimensiones: año, mes, trimestre
   Uso: Tendencias y forecasting

📊 KPI_2_TOP_PRODUCTOS:
   Descripción: Top productos y categorías más rentables
   Métricas: ventas, ganancia, cantidad_vendida, num_ordenes
   Dimensiones: producto, categoria, subcategoria
   Uso: Análisis de portafolio de productos

📊 KPI_3_ANALISIS_CLIENTES:
   Descripción: Segmentación y comportamiento de clientes
   Métricas: ventas_por_cliente, ganancia_por_cliente, frecuencia_compra
   Dimensiones: segmento, region
   Uso: Customer analytics y targeting

📊 KPI_4_PERFORMANCE_GEOGRAFICO:
   Descripción: Desempeño por región y estado
   Métricas: ventas, ganancia, margen, ordenes
   Dimensiones: region, estado, ciudad
   Uso: Expansión geográfica y estrategia regional

📊 KPI_5_EFICIENC

In [0]:
print("\n" + "="*80)
print("KPI 1: VENTAS Y RENTABILIDAD TEMPORAL")
print("="*80)

# Unir fact table con dimensión tiempo
df_temporal = fact_table.merge(
    dimensions['dim_tiempo'],
    on='tiempo_id',
    how='left'
)

# KPI Mensual
kpi_mensual = df_temporal.groupby(['año', 'mes']).agg({
    'ventas': ['sum', 'mean', 'count'],
    'ganancia': ['sum', 'mean'],
    'cantidad': 'sum',
    'descuento': 'mean'
}).reset_index()

# Aplanar nombres de columnas
kpi_mensual.columns = ['año', 'mes', 'ventas_totales', 'ventas_promedio', 'num_ordenes', 
                        'ganancia_total', 'ganancia_promedio', 'cantidad_total', 'descuento_promedio']

# Calcular métricas adicionales
kpi_mensual['ticket_promedio'] = kpi_mensual['ventas_totales'] / kpi_mensual['num_ordenes']
kpi_mensual['margen_porcentaje'] = (kpi_mensual['ganancia_total'] / kpi_mensual['ventas_totales'] * 100).round(2)

# Ordenar por fecha
kpi_mensual = kpi_mensual.sort_values(['año', 'mes'])

# Calcular crecimiento mes a mes
kpi_mensual['crecimiento_ventas_pct'] = kpi_mensual['ventas_totales'].pct_change() * 100
kpi_mensual['crecimiento_ganancia_pct'] = kpi_mensual['ganancia_total'].pct_change() * 100

# Acumulado anual
kpi_mensual['ventas_acum_año'] = kpi_mensual.groupby('año')['ventas_totales'].cumsum()
kpi_mensual['ganancia_acum_año'] = kpi_mensual.groupby('año')['ganancia_total'].cumsum()

logger.info(f"✓ KPI Mensual creado: {len(kpi_mensual)} registros")
display(kpi_mensual.tail(12))

# KPI Anual
kpi_anual = df_temporal.groupby('año').agg({
    'ventas': ['sum', 'mean'],
    'ganancia': ['sum', 'mean'],
    'cantidad': 'sum',
    'order_id': 'nunique'
}).reset_index()

kpi_anual.columns = ['año', 'ventas_totales', 'ventas_promedio', 'ganancia_total', 
                      'ganancia_promedio', 'cantidad_total', 'num_ordenes_unicas']

kpi_anual['margen_porcentaje'] = (kpi_anual['ganancia_total'] / kpi_anual['ventas_totales'] * 100).round(2)
kpi_anual['crecimiento_anual_pct'] = kpi_anual['ventas_totales'].pct_change() * 100

logger.info(f"✓ KPI Anual creado: {len(kpi_anual)} registros")
display(kpi_anual)

# Guardar en Gold
path_kpi_mensual = f"{gold_path}kpi_ventas_mensual/"
wr.s3.to_parquet(df=kpi_mensual, path=path_kpi_mensual, dataset=True, mode='overwrite')
logger.info(f"✓ KPI Mensual guardado en: {path_kpi_mensual}")

path_kpi_anual = f"{gold_path}kpi_ventas_anual/"
wr.s3.to_parquet(df=kpi_anual, path=path_kpi_anual, dataset=True, mode='overwrite')
logger.info(f"✓ KPI Anual guardado en: {path_kpi_anual}")




KPI 1: VENTAS Y RENTABILIDAD TEMPORAL


,año,mes,ventas_totales,ventas_promedio,num_ordenes,ganancia_total,ganancia_promedio,cantidad_total,descuento_promedio,ticket_promedio,margen_porcentaje,crecimiento_ventas_pct,crecimiento_ganancia_pct,ventas_acum_año,ganancia_acum_año
36,2017,1,15517.6080,110.840057,140,1555.1549,11.108249,511,0.155714,110.840057,10.02,-64.605459,-76.492166,15517.6080,1555.1549
37,2017,2,14352.7974,138.007667,104,1692.6118,16.275113,331,0.169904,138.007667,11.79,-7.506380,8.838792,29870.4054,3247.7667
38,2017,3,27240.6718,120.533946,226,4596.2667,20.337463,829,0.132478,120.533946,16.87,89.793467,171.548780,57111.0772,7844.0334
39,2017,4,21204.0986,106.553259,199,1966.6991,9.882910,700,0.194573,106.553259,9.28,-22.160148,-57.210945,78315.1758,9810.7325
40,2017,5,27302.3152,118.705718,230,3369.6116,14.650485,811,0.150609,118.705718,12.34,28.759613,71.333358,105617.4910,13180.3441
41,2017,6,30277.2397,128.293389,236,3481.1988,14.750842,851,0.170636,128.293389,11.50,10.896235,3.311575,135894.7307,16661.5429
42,2017,7,26940.6370,124.725171,216,4102.4552,18.992848,768,0.131481,124.725171,15.23,-11.020168,17.846048,162835.3677,20763.9981
43,2017,8,21586.6020,109.023242,198,3777.5010,19.078288,737,0.143434,109.023242,17.50,-19.873454,-7.920969,184421.9697,24541.4991
44,2017,9,43153.6360,99.892676,432,7000.8533,16.205679,1514,0.139468,99.892676,16.22,99.909351,85.330283,227575.6057,31542.3524
45,2017,10,31527.8692,112.599533,280,2686.6470,9.595168,997,0.167571,112.599533,8.52,-26.940411,-61.624007,259103.4749,34228.9994


,año,ventas_totales,ventas_promedio,ganancia_total,ganancia_promedio,cantidad_total,num_ordenes_unicas,margen_porcentaje,crecimiento_anual_pct
0,2014,227443.0151,122.215484,27126.8213,14.576476,6823,923,11.93,NaN
1,2015,239391.0360,122.450658,31993.1148,16.364765,7097,997,13.36,5.253193
2,2016,289623.1343,119.580155,36308.7481,14.991225,8889,1257,12.54,20.983283
3,2017,358537.3837,114.915828,46142.2098,14.789170,11256,1613,12.87,23.794456


In [0]:
print("\n" + "="*80)
print("KPI 2: TOP PRODUCTOS Y CATEGORÍAS")
print("="*80)

# Unir con dimensión producto
df_productos = fact_table.merge(
    dimensions['dim_producto'],
    on='producto_id',
    how='left'
)

# Top 20 Productos
kpi_top_productos = df_productos.groupby(['producto_id', 'nombre_producto', 'categoria', 'subcategoria']).agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum',
    'order_id': 'count'
}).reset_index()

kpi_top_productos.columns = ['producto_id', 'nombre_producto', 'categoria', 'subcategoria', 
                               'ventas_totales', 'ganancia_total', 'cantidad_vendida', 'num_ordenes']

kpi_top_productos = kpi_top_productos.sort_values('ventas_totales', ascending=False).head(20)
kpi_top_productos['ranking'] = range(1, len(kpi_top_productos) + 1)
kpi_top_productos['participacion_ventas_pct'] = (kpi_top_productos['ventas_totales'] / kpi_top_productos['ventas_totales'].sum() * 100).round(2)
kpi_top_productos['margen_pct'] = (kpi_top_productos['ganancia_total'] / kpi_top_productos['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI Top 20 Productos creado")
display(kpi_top_productos)

# Top Categorías
kpi_categorias = df_productos.groupby('categoria').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum',
    'order_id': 'count'
}).reset_index()

kpi_categorias.columns = ['categoria', 'ventas_totales', 'ganancia_total', 'cantidad_total', 'num_ordenes']
kpi_categorias = kpi_categorias.sort_values('ventas_totales', ascending=False)
kpi_categorias['participacion_pct'] = (kpi_categorias['ventas_totales'] / kpi_categorias['ventas_totales'].sum() * 100).round(2)
kpi_categorias['margen_pct'] = (kpi_categorias['ganancia_total'] / kpi_categorias['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI Categorías creado")
display(kpi_categorias)

# Top Subcategorías
kpi_subcategorias = df_productos.groupby(['categoria', 'subcategoria']).agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum'
}).reset_index()

kpi_subcategorias.columns = ['categoria', 'subcategoria', 'ventas_totales', 'ganancia_total', 'cantidad_total']
kpi_subcategorias = kpi_subcategorias.sort_values('ventas_totales', ascending=False).head(15)
kpi_subcategorias['ranking'] = range(1, len(kpi_subcategorias) + 1)

logger.info(f"✓ KPI Subcategorías creado")

# Guardar en Gold
gold_bucket = "sin-exam-gold-877617909831"
path_top_productos = f"s3://{gold_bucket}/kpi_top_productos/"
wr.s3.to_parquet(df=kpi_top_productos, path=path_top_productos, dataset=True, mode='overwrite')

path_categorias = f"s3://{gold_bucket}/kpi_categorias/"
wr.s3.to_parquet(df=kpi_categorias, path=path_categorias, dataset=True, mode='overwrite')

path_subcategorias = f"s3://{gold_bucket}/kpi_subcategorias/"
wr.s3.to_parquet(df=kpi_subcategorias, path=path_subcategorias, dataset=True, mode='overwrite')

logger.info("✓ KPIs de productos guardados en Gold")




KPI 2: TOP PRODUCTOS Y CATEGORÍAS


,producto_id,nombre_producto,categoria,subcategoria,ventas_totales,ganancia_total,cantidad_vendida,num_ordenes,ranking,participacion_ventas_pct,margen_pct
837,838,Tennsco Double-Tier Lockers,Office Supplies,Storage,6705.596,49.5044,34,10,1,7.42,0.74
204,205,Global Leather Highback Executive Chair with P...,Furniture,Chairs,5989.204,267.3034,39,11,2,6.62,4.46
1122,1123,SAFCO Arco Folding Chair,Furniture,Chairs,5109.700,403.2520,24,8,3,5.65,7.89
744,745,"Global High-Back Leather Tilter, Burgundy",Furniture,Chairs,4833.507,-311.1647,47,13,4,5.35,-6.44
1593,1594,Fellowes Staxonsteel Drawer Files,Office Supplies,Storage,4829.250,509.9688,26,7,5,5.34,10.56
448,449,Hon Every-Day Series Multi-Task Chairs,Furniture,Chairs,4473.924,-37.5960,30,8,6,4.95,-0.84
1287,1288,"Tennsco Snap-Together Open Shelving Units, Sta...",Office Supplies,Storage,4471.680,223.5840,16,4,7,4.95,5.00
1111,1112,Global Commerce Series Low-Back Swivel/Tilt Ch...,Furniture,Chairs,4420.056,41.1168,24,7,8,4.89,0.93
244,245,"Tennsco Stur-D-Stor Boltless Shelving, 5 Shelv...",Office Supplies,Storage,4384.044,-36.5337,33,7,9,4.85,-0.83
29,30,GE 30524EE4,Technology,Phones,4311.780,554.6517,27,8,10,4.77,12.86


,categoria,ventas_totales,ganancia_total,cantidad_total,num_ordenes,participacion_pct,margen_pct
1,Office Supplies,399842.0650,78929.4453,21929,5878,35.86,19.74
0,Furniture,393451.2071,17755.3680,6541,1895,35.29,4.51
2,Technology,321701.2970,44886.0807,5595,1585,28.85,13.95


In [0]:

print("\n" + "="*80)
print("KPI 3: ANÁLISIS DE CLIENTES Y SEGMENTACIÓN")
print("="*80)

# Unir con dimensiones cliente y ubicación
df_clientes = fact_table.merge(
    dimensions['dim_cliente'],
    on='cliente_id',
    how='left'
).merge(
    dimensions['dim_ubicacion'],
    on='ubicacion_id',
    how='left'
)

# KPI por Segmento de Cliente
kpi_segmento = df_clientes.groupby('segmento').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum',
    'cliente_id': 'nunique',
    'order_id': 'count'
}).reset_index()

kpi_segmento.columns = ['segmento', 'ventas_totales', 'ganancia_total', 'cantidad_total', 
                         'num_clientes', 'num_ordenes']

kpi_segmento['ventas_por_cliente'] = (kpi_segmento['ventas_totales'] / kpi_segmento['num_clientes']).round(2)
kpi_segmento['ganancia_por_cliente'] = (kpi_segmento['ganancia_total'] / kpi_segmento['num_clientes']).round(2)
kpi_segmento['ordenes_por_cliente'] = (kpi_segmento['num_ordenes'] / kpi_segmento['num_clientes']).round(2)
kpi_segmento['participacion_ventas_pct'] = (kpi_segmento['ventas_totales'] / kpi_segmento['ventas_totales'].sum() * 100).round(2)
kpi_segmento['margen_pct'] = (kpi_segmento['ganancia_total'] / kpi_segmento['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI por Segmento creado")
display(kpi_segmento)

# KPI por Región
kpi_region = df_clientes.groupby('region').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum',
    'order_id': 'count'
}).reset_index()

kpi_region.columns = ['region', 'ventas_totales', 'ganancia_total', 'cantidad_total', 'num_ordenes']
kpi_region = kpi_region.sort_values('ventas_totales', ascending=False)
kpi_region['participacion_ventas_pct'] = (kpi_region['ventas_totales'] / kpi_region['ventas_totales'].sum() * 100).round(2)
kpi_region['margen_pct'] = (kpi_region['ganancia_total'] / kpi_region['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI por Región creado")
display(kpi_region)

# Top 10 Estados
kpi_estados = df_clientes.groupby('estado').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'order_id': 'count'
}).reset_index()

kpi_estados.columns = ['estado', 'ventas_totales', 'ganancia_total', 'num_ordenes']
kpi_estados = kpi_estados.sort_values('ventas_totales', ascending=False).head(10)
kpi_estados['ranking'] = range(1, len(kpi_estados) + 1)
kpi_estados['margen_pct'] = (kpi_estados['ganancia_total'] / kpi_estados['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI Top 10 Estados creado")

# Guardar en Gold
path_segmento = f"s3://{gold_bucket}/kpi_segmento_cliente/"
wr.s3.to_parquet(df=kpi_segmento, path=path_segmento, dataset=True, mode='overwrite')

path_region = f"s3://{gold_bucket}/kpi_region/"
wr.s3.to_parquet(df=kpi_region, path=path_region, dataset=True, mode='overwrite')

path_estados = f"s3://{gold_bucket}/kpi_top_estados/"
wr.s3.to_parquet(df=kpi_estados, path=path_estados, dataset=True, mode='overwrite')

logger.info("✓ KPIs de clientes y geografía guardados en Gold")




KPI 3: ANÁLISIS DE CLIENTES Y SEGMENTACIÓN


,segmento,ventas_totales,ganancia_total,cantidad_total,num_clientes,num_ordenes,ventas_por_cliente,ganancia_por_cliente,ordenes_por_cliente,participacion_ventas_pct,margen_pct
0,Consumer,584208.9250,71685.5016,17755,409,4901,1428.38,175.27,11.98,52.40,12.27
1,Corporate,332092.8101,42764.8011,10290,234,2780,1419.20,182.76,11.88,29.78,12.88
2,Home Office,198692.8340,27120.5913,6020,148,1677,1342.52,183.25,11.33,17.82,13.65


,region,ventas_totales,ganancia_total,cantidad_total,num_ordenes,participacion_ventas_pct,margen_pct
3,West,411857.5455,59598.9893,11321,3058,36.94,14.47
1,East,288054.0180,36512.4387,9345,2634,25.83,12.68
0,Central,244869.7356,19758.8110,7886,2160,21.96,8.07
2,South,170213.2700,25700.6550,5513,1506,15.27,15.10


In [0]:

print("\n" + "="*80)
print("KPI 4: MÉTRICAS GLOBALES EJECUTIVAS")
print("="*80)

# Calcular métricas globales
kpi_global = pd.DataFrame({
    'metrica': [
        'Ventas Totales',
        'Ganancia Total',
        'Margen Promedio (%)',
        'Número de Órdenes',
        'Número de Clientes',
        'Ticket Promedio',
        'Cantidad Total Vendida',
        'Descuento Promedio (%)',
        'Ventas Promedio por Orden',
        'Ganancia Promedio por Orden'
    ],
    'valor': [
        fact_table['ventas'].sum(),
        fact_table['ganancia'].sum(),
        (fact_table['ganancia'].sum() / fact_table['ventas'].sum() * 100),
        fact_table['order_id'].nunique(),
        fact_table['cliente_id'].nunique(),
        fact_table['ventas'].sum() / fact_table['order_id'].nunique(),
        fact_table['cantidad'].sum(),
        fact_table['descuento'].mean() * 100,
        fact_table['ventas'].mean(),
        fact_table['ganancia'].mean()
    ]
})

kpi_global['valor'] = kpi_global['valor'].round(2)

# Formatear valores
def format_value(row):
    if 'Número' in row['metrica']:
        return f"{int(row['valor']):,}"
    elif '%' in row['metrica']:
        return f"{row['valor']:.2f}%"
    elif 'Total' in row['metrica'] or 'Promedio' in row['metrica'] or 'Ticket' in row['metrica']:
        return f"${row['valor']:,.2f}"
    else:
        return f"{row['valor']:,.2f}"

kpi_global['valor_formateado'] = kpi_global.apply(format_value, axis=1)

logger.info("✓ KPI Global creado")
display(kpi_global)

# Métricas adicionales de rentabilidad
kpi_rentabilidad = pd.DataFrame({
    'metrica': [
        'Órdenes Rentables',
        'Órdenes con Pérdida',
        '% Órdenes Rentables',
        'Ganancia Promedio (Órdenes Rentables)',
        'Pérdida Promedio (Órdenes con Pérdida)'
    ],
    'valor': [
        len(fact_table[fact_table['ganancia'] > 0]),
        len(fact_table[fact_table['ganancia'] < 0]),
        (len(fact_table[fact_table['ganancia'] > 0]) / len(fact_table) * 100),
        fact_table[fact_table['ganancia'] > 0]['ganancia'].mean(),
        fact_table[fact_table['ganancia'] < 0]['ganancia'].mean()
    ]
})

kpi_rentabilidad['valor'] = kpi_rentabilidad['valor'].round(2)

logger.info("✓ KPI Rentabilidad creado")
display(kpi_rentabilidad)

# Guardar en Gold
path_global = f"s3://{gold_bucket}/kpi_metricas_globales/"
wr.s3.to_parquet(df=kpi_global, path=path_global, dataset=True, mode='overwrite')

path_rentabilidad = f"s3://{gold_bucket}/kpi_rentabilidad/"
wr.s3.to_parquet(df=kpi_rentabilidad, path=path_rentabilidad, dataset=True, mode='overwrite')

logger.info("✓ KPIs globales guardados en Gold")




KPI 4: MÉTRICAS GLOBALES EJECUTIVAS


,metrica,valor,valor_formateado
0,Ventas Totales,1114994.57,"$1,114,994.57"
1,Ganancia Total,141570.89,"$141,570.89"
2,Margen Promedio (%),12.70,12.70%
3,Número de Órdenes,4790.00,"4,790"
4,Número de Clientes,791.00,791
5,Ticket Promedio,232.78,$232.78
6,Cantidad Total Vendida,34065.00,"$34,065.00"
7,Descuento Promedio (%),15.21,15.21%
8,Ventas Promedio por Orden,119.15,$119.15
9,Ganancia Promedio por Orden,15.13,$15.13


,metrica,valor
0,Órdenes Rentables,7657.00
1,Órdenes con Pérdida,1639.00
2,% Órdenes Rentables,81.82
3,Ganancia Promedio (Órdenes Rentables),23.74
4,Pérdida Promedio (Órdenes con Pérdida),-24.51


In [0]:
print("\n" + "="*80)
print("KPI 5: ANÁLISIS OPERACIONAL (ENVÍOS Y DESCUENTOS)")
print("="*80)

# Unir con dimensión envío
df_envios = fact_table.merge(
    dimensions['dim_envio'],
    on='envio_id',
    how='left'
)

# KPI por Modo de Envío
kpi_envio = df_envios.groupby('modo_envio').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'order_id': 'count',
    'cantidad': 'sum'
}).reset_index()

kpi_envio.columns = ['modo_envio', 'ventas_totales', 'ganancia_total', 'num_ordenes', 'cantidad_total']
kpi_envio = kpi_envio.sort_values('num_ordenes', ascending=False)
kpi_envio['participacion_ordenes_pct'] = (kpi_envio['num_ordenes'] / kpi_envio['num_ordenes'].sum() * 100).round(2)
kpi_envio['ventas_promedio_por_orden'] = (kpi_envio['ventas_totales'] / kpi_envio['num_ordenes']).round(2)
kpi_envio['margen_pct'] = (kpi_envio['ganancia_total'] / kpi_envio['ventas_totales'] * 100).round(2)

logger.info(f"✓ KPI por Modo de Envío creado")
display(kpi_envio)

# Análisis de Descuentos
# Crear rangos de descuento
df_descuentos = fact_table.copy()
df_descuentos['rango_descuento'] = pd.cut(
    df_descuentos['descuento'], 
    bins=[0, 0.1, 0.2, 0.3, 1.0],
    labels=['0-10%', '10-20%', '20-30%', '>30%'],
    include_lowest=True
)

kpi_descuentos = df_descuentos.groupby('rango_descuento').agg({
    'ventas': 'sum',
    'ganancia': 'sum',
    'order_id': 'count',
    'descuento': 'mean'
}).reset_index()

kpi_descuentos.columns = ['rango_descuento', 'ventas_totales', 'ganancia_total', 
                           'num_ordenes', 'descuento_promedio']

kpi_descuentos['margen_pct'] = (kpi_descuentos['ganancia_total'] / kpi_descuentos['ventas_totales'] * 100).round(2)
kpi_descuentos['descuento_promedio_pct'] = (kpi_descuentos['descuento_promedio'] * 100).round(2)

logger.info(f"✓ KPI de Descuentos creado")
display(kpi_descuentos)

# Matriz: Categoría vs Modo de Envío
df_matriz = fact_table.merge(
    dimensions['dim_producto'][['producto_id', 'categoria']],
    on='producto_id',
    how='left'
).merge(
    dimensions['dim_envio'],
    on='envio_id',
    how='left'
)

kpi_matriz_categoria_envio = df_matriz.groupby(['categoria', 'modo_envio']).agg({
    'ventas': 'sum',
    'order_id': 'count'
}).reset_index()

kpi_matriz_categoria_envio.columns = ['categoria', 'modo_envio', 'ventas_totales', 'num_ordenes']

logger.info(f"✓ Matriz Categoría x Envío creada")

# Guardar en Gold
path_envio = f"s3://{gold_bucket}/kpi_modo_envio/"
wr.s3.to_parquet(df=kpi_envio, path=path_envio, dataset=True, mode='overwrite')

path_descuentos = f"s3://{gold_bucket}/kpi_descuentos/"
wr.s3.to_parquet(df=kpi_descuentos, path=path_descuentos, dataset=True, mode='overwrite')

path_matriz = f"s3://{gold_bucket}/kpi_matriz_categoria_envio/"
wr.s3.to_parquet(df=kpi_matriz_categoria_envio, path=path_matriz, dataset=True, mode='overwrite')

logger.info("✓ KPIs operacionales guardados en Gold")




KPI 5: ANÁLISIS OPERACIONAL (ENVÍOS Y DESCUENTOS)


,modo_envio,ventas_totales,ganancia_total,num_ordenes,cantidad_total,participacion_ordenes_pct,ventas_promedio_por_orden,margen_pct
3,Standard Class,662532.3994,82476.5490,5589,20541,59.72,118.54,12.45
2,Second Class,220491.0832,29425.6010,1806,6566,19.30,122.09,13.35
0,First Class,169417.3705,20845.3899,1448,5163,15.47,117.00,12.30
1,Same Day,62553.7160,8823.3541,515,1795,5.50,121.46,14.11


/tmp/ipykernel_82/892135363.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  kpi_descuentos = df_descuentos.groupby('rango_descuento').agg({


,rango_descuento,ventas_totales,ganancia_total,num_ordenes,descuento_promedio,margen_pct,descuento_promedio_pct
0,0-10%,477663.1610,122829.9922,4552,0.001670,25.71,0.17
1,10-20%,502525.1985,46585.4504,3640,0.199341,9.27,19.93
2,20-30%,73285.6810,-6313.7608,203,0.300000,-8.62,30.00
3,>30%,61520.5286,-21530.7878,963,0.653188,-35.00,65.32


In [0]:

print("\n" + "="*80)
print("KPI 6: FORECASTING Y ANÁLISIS DE TENDENCIAS")
print("="*80)

# Usar datos mensuales para forecasting
kpi_forecast = kpi_mensual.copy()

# Calcular media móvil (3 meses)
kpi_forecast['media_movil_3m'] = kpi_forecast['ventas_totales'].rolling(window=3).mean()

# Calcular media móvil (6 meses)
kpi_forecast['media_movil_6m'] = kpi_forecast['ventas_totales'].rolling(window=6).mean()

# Regresión lineal para tendencia
kpi_forecast['periodo'] = range(len(kpi_forecast))

from sklearn.linear_model import LinearRegression

X = kpi_forecast['periodo'].values.reshape(-1, 1)
y_ventas = kpi_forecast['ventas_totales'].values
y_ganancia = kpi_forecast['ganancia_total'].values

# Modelo para ventas
model_ventas = LinearRegression()
model_ventas.fit(X, y_ventas)
kpi_forecast['tendencia_ventas'] = model_ventas.predict(X)

# Modelo para ganancia
model_ganancia = LinearRegression()
model_ganancia.fit(X, y_ganancia)
kpi_forecast['tendencia_ganancia'] = model_ganancia.predict(X)

logger.info("✓ Tendencias calculadas con regresión lineal")
display(kpi_forecast[['año', 'mes', 'ventas_totales', 'media_movil_3m', 'tendencia_ventas']].tail(12))

# Forecast próximos 6 meses
future_periods = np.array(range(len(kpi_forecast), len(kpi_forecast) + 6)).reshape(-1, 1)
forecast_ventas = model_ventas.predict(future_periods)
forecast_ganancia = model_ganancia.predict(future_periods)

# Crear dataframe de forecast
last_year = kpi_forecast['año'].iloc[-1]
last_month = kpi_forecast['mes'].iloc[-1]

forecast_months = []
for i in range(1, 7):
    new_month = (last_month + i - 1) % 12 + 1
    new_year = last_year + (last_month + i - 1) // 12
    forecast_months.append((new_year, new_month))

kpi_forecast_future = pd.DataFrame({
    'año': [y for y, m in forecast_months],
    'mes': [m for y, m in forecast_months],
    'ventas_forecast': forecast_ventas,
    'ganancia_forecast': forecast_ganancia,
    'tipo': 'Proyección'
})

logger.info("✓ Forecast para próximos 6 meses creado")
display(kpi_forecast_future)

# Identificar estacionalidad
kpi_estacionalidad = kpi_mensual.groupby('mes').agg({
    'ventas_totales': ['mean', 'std', 'count']
}).reset_index()

kpi_estacionalidad.columns = ['mes', 'ventas_promedio', 'desviacion_std', 'num_años']
kpi_estacionalidad = kpi_estacionalidad.sort_values('ventas_promedio', ascending=False)
kpi_estacionalidad['ranking'] = range(1, len(kpi_estacionalidad) + 1)

# Nombrar meses
month_names = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
               7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
kpi_estacionalidad['nombre_mes'] = kpi_estacionalidad['mes'].map(month_names)

logger.info("✓ Análisis de estacionalidad completado")
display(kpi_estacionalidad)

# Guardar en Gold
path_forecast = f"s3://{gold_bucket}/kpi_tendencias_historicas/"
wr.s3.to_parquet(df=kpi_forecast, path=path_forecast, dataset=True, mode='overwrite')

path_future = f"s3://{gold_bucket}/kpi_forecast_futuro/"
wr.s3.to_parquet(df=kpi_forecast_future, path=path_future, dataset=True, mode='overwrite')

path_estacionalidad = f"s3://{gold_bucket}/kpi_estacionalidad/"
wr.s3.to_parquet(df=kpi_estacionalidad, path=path_estacionalidad, dataset=True, mode='overwrite')

logger.info("✓ KPIs de forecasting y estacionalidad guardados en Gold")
print("\n" + "="*80)
print("CATALOGANDO KPIS EN GLUE")
print("="*80)

glue = boto3.client('glue', region_name="us-east-1")
crawler_name = "sin-exam-gold-crawler"

logger.info(f"Ejecutando crawler: {crawler_name}")
try:
    glue.start_crawler(Name=crawler_name)
    logger.info("✓ Crawler iniciado - tardará 1-2 minutos")
except Exception as e:
    if 'CrawlerRunningException' in str(e):
        logger.warning("⚠️  Crawler ya está corriendo")
    else:
        logger.error(f"Error: {e}")




KPI 6: FORECASTING Y ANÁLISIS DE TENDENCIAS


,año,mes,ventas_totales,media_movil_3m,tendencia_ventas
36,2017,1,15517.6080,31860.651000,28882.961744
37,2017,2,14352.7974,24570.739133,29335.274402
38,2017,3,27240.6718,19037.025733,29787.587059
39,2017,4,21204.0986,20932.522600,30239.899717
40,2017,5,27302.3152,25249.028533,30692.212375
41,2017,6,30277.2397,26261.217833,31144.525032
42,2017,7,26940.6370,28173.397300,31596.837690
43,2017,8,21586.6020,26268.159567,32049.150348
44,2017,9,43153.6360,30560.291667,32501.463005
45,2017,10,31527.8692,32089.369067,32953.775663


,año,mes,ventas_forecast,ganancia_forecast,tipo
0,2018,1,34310.713636,4440.093120,Proyección
1,2018,2,34763.026294,4500.937997,Proyección
2,2018,3,35215.338951,4561.782874,Proyección
3,2018,4,35667.651609,4622.627752,Proyección
4,2018,5,36119.964267,4683.472629,Proyección
5,2018,6,36572.276924,4744.317506,Proyección


,mes,ventas_promedio,desviacion_std,num_años,ranking,nombre_mes
10,11,42166.769675,5839.644968,4,1,Noviembre
11,12,41222.436125,6782.006814,4,2,Diciembre
8,9,35229.422325,7853.975328,4,3,Septiembre
9,10,22162.536550,6245.270048,4,4,Octubre
6,7,22098.521250,4585.813185,4,5,Julio
5,6,20588.096075,7626.189754,4,6,Junio
7,8,19447.434875,1776.502651,4,7,Agosto
2,3,19380.424600,6067.138035,4,8,Marzo
4,5,19379.235425,6590.842680,4,9,Mayo
3,4,18859.435025,2292.596604,4,10,Abril



CATALOGANDO KPIS EN GLUE


In [0]:
print("\n" + "="*80)
print("RESUMEN FINAL DE KPIS CREADOS")
print("="*80)

kpi_summary = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'SampleSuperstore',
    'total_registros_fact': len(fact_table),
    'kpis_creados': [
        {
            'nombre': 'KPI Ventas Mensual',
            'registros': len(kpi_mensual),
            'path': f"s3://{gold_bucket}/kpi_ventas_mensual/",
            'metricas': ['ventas_totales', 'ganancia_total', 'margen_%', 'crecimiento_%']
        },
        {
            'nombre': 'KPI Ventas Anual',
            'registros': len(kpi_anual),
            'path': f"s3://{gold_bucket}/kpi_ventas_anual/",
            'metricas': ['ventas_totales', 'ganancia_total', 'margen_%']
        },
        {
            'nombre': 'KPI Top 20 Productos',
            'registros': len(kpi_top_productos),
            'path': f"s3://{gold_bucket}/kpi_top_productos/",
            'metricas': ['ventas_totales', 'ganancia_total', 'participacion_%']
        },
        {
            'nombre': 'KPI Categorías',
            'registros': len(kpi_categorias),
            'path': f"s3://{gold_bucket}/kpi_categorias/",
            'metricas': ['ventas_totales', 'ganancia_total', 'margen_%']
        },
        {
            'nombre': 'KPI Subcategorías',
            'registros': len(kpi_subcategorias),
            'path': f"s3://{gold_bucket}/kpi_subcategorias/",
            'metricas': ['ventas_totales', 'ganancia_total']
        },
        {
            'nombre': 'KPI Segmento Cliente',
            'registros': len(kpi_segmento),
            'path': f"s3://{gold_bucket}/kpi_segmento_cliente/",
            'metricas': ['ventas_por_cliente', 'ganancia_por_cliente', 'ordenes_por_cliente']
        },
        {
            'nombre': 'KPI Región',
            'registros': len(kpi_region),
            'path': f"s3://{gold_bucket}/kpi_region/",
            'metricas': ['ventas_totales', 'ganancia_total', 'margen_%']
        },
        {
            'nombre': 'KPI Top 10 Estados',
            'registros': len(kpi_estados),
            'path': f"s3://{gold_bucket}/kpi_top_estados/",
            'metricas': ['ventas_totales', 'ganancia_total']
        },
        {
            'nombre': 'KPI Métricas Globales',
            'registros': len(kpi_global),
            'path': f"s3://{gold_bucket}/kpi_metricas_globales/",
            'metricas': ['KPI cards para dashboard ejecutivo']
        },
        {
            'nombre': 'KPI Modo de Envío',
            'registros': len(kpi_envio),
            'path': f"s3://{gold_bucket}/kpi_modo_envio/",
            'metricas': ['ventas_totales', 'num_ordenes', 'participacion_%']
        },
        {
            'nombre': 'KPI Descuentos',
            'registros': len(kpi_descuentos),
            'path': f"s3://{gold_bucket}/kpi_descuentos/",
            'metricas': ['ventas_totales', 'ganancia_total', 'margen_%']
        },
        {
            'nombre': 'KPI Tendencias Históricas',
            'registros': len(kpi_forecast),
            'path': f"s3://{gold_bucket}/kpi_tendencias_historicas/",
            'metricas': ['media_movil_3m', 'media_movil_6m', 'tendencia_lineal']
        },
        {
            'nombre': 'KPI Forecast Futuro',
            'registros': len(kpi_forecast_future),
            'path': f"s3://{gold_bucket}/kpi_forecast_futuro/",
            'metricas': ['ventas_forecast', 'ganancia_forecast']
        },
        {
            'nombre': 'KPI Estacionalidad',
            'registros': len(kpi_estacionalidad),
            'path': f"s3://{gold_bucket}/kpi_estacionalidad/",
            'metricas': ['ventas_promedio_por_mes', 'ranking']
        }
    ]
}

print("\n📊 RESUMEN:")
print(json.dumps(kpi_summary, indent=2))

# Guardar resumen
s3 = boto3.client('s3')
summary_json = json.dumps(kpi_summary, indent=2)
s3.put_object(
    Bucket=gold_bucket,
    Key='kpi_summary.json',
    Body=summary_json,
    ContentType='application/json'
)

print("\n✓ Resumen guardado en S3")

# Ejecutar crawler Gold
glue = boto3.client('glue', region_name="us-east-1")
crawler_name = "sin-exam-gold-crawler"

logger.info(f"Ejecutando crawler: {crawler_name}")
try:
    glue.start_crawler(Name=crawler_name)
    logger.info("✓ Crawler iniciado - tardará 1-2 minutos")
except Exception as e:
    if 'CrawlerRunningException' in str(e):
        logger.warning("⚠️  Crawler ya está corriendo")
    else:
        logger.error(f"Error: {e}")

print("\n" + "="*80)
print("CAPA GOLD COMPLETADA - SUPERSTORE")
print("="*80)
print("\n🎯 PRÓXIMOS PASOS:")
print("1. Espera 2 minutos a que el crawler termine")
print("2. Verifica en Athena que puedes consultar los KPIs:")
print(f"   SELECT * FROM {database}.gold_kpi_ventas_mensual ORDER BY año DESC, mes DESC LIMIT 12;")
print(f"   SELECT * FROM {database}.gold_kpi_categorias;")
print(f"   SELECT * FROM {database}.gold_kpi_metricas_globales;")
print("3. Configura QuickSight para conectarse a Athena")
print("4. Crea dashboards usando las tablas Gold")

print("\n📊 TABLAS GOLD DISPONIBLES PARA QUICKSIGHT:")
print(f"   • {database}.gold_kpi_ventas_mensual (tendencias temporales)")
print(f"   • {database}.gold_kpi_ventas_anual (resumen anual)")
print(f"   • {database}.gold_kpi_top_productos (top 20 productos)")
print(f"   • {database}.gold_kpi_categorias (análisis por categoría)")
print(f"   • {database}.gold_kpi_subcategorias (análisis por subcategoría)")
print(f"   • {database}.gold_kpi_segmento_cliente (segmentación de clientes)")
print(f"   • {database}.gold_kpi_region (análisis geográfico)")
print(f"   • {database}.gold_kpi_top_estados (top 10 estados)")
print(f"   • {database}.gold_kpi_metricas_globales (KPI cards)")
print(f"   • {database}.gold_kpi_modo_envio (análisis de logística)")
print(f"   • {database}.gold_kpi_descuentos (impacto de descuentos)")
print(f"   • {database}.gold_kpi_tendencias_historicas (con media móvil)")
print(f"   • {database}.gold_kpi_forecast_futuro (proyecciones 6 meses)")
print(f"   • {database}.gold_kpi_estacionalidad (patrones mensuales)")

print("\n💡 SUGERENCIAS PARA DASHBOARDS QUICKSIGHT:")
print("\nDASHBOARD 1 - EXECUTIVE OVERVIEW:")
print("  • KPI Cards: kpi_metricas_globales (4-6 cards)")
print("  • Line Chart: kpi_ventas_mensual (tendencia con forecast)")
print("  • Bar Chart: kpi_categorias (ventas por categoría)")
print("  • Pie Chart: kpi_segmento_cliente (distribución)")
print("  • Filtros: Año, Región")

print("\nDASHBOARD 2 - DETAILED ANALYSIS:")
print("  • Heatmap: kpi_matriz_categoria_envio (categoría vs envío)")
print("  • Top N Table: kpi_top_productos (con drill-down)")
print("  • Geo Map: kpi_region + kpi_top_estados")
print("  • Combo Chart: kpi_ventas_mensual (ventas + crecimiento %)")
print("  • Bar Chart: kpi_descuentos (impacto del descuento)")
print("  • Filtros: Año, Mes, Categoría, Región, Segmento")

print("\n✅ ¡CAPA GOLD LISTA PARA VISUALIZACIÓN!")


RESUMEN FINAL DE KPIS CREADOS

📊 RESUMEN:
{
  "timestamp": "2025-12-16T06:34:47.165012",
  "dataset": "SampleSuperstore",
  "total_registros_fact": 9358,
  "kpis_creados": [
    {
      "nombre": "KPI Ventas Mensual",
      "registros": 48,
      "path": "s3://sin-exam-gold-877617909831/kpi_ventas_mensual/",
      "metricas": [
        "ventas_totales",
        "ganancia_total",
        "margen_%",
        "crecimiento_%"
      ]
    },
    {
      "nombre": "KPI Ventas Anual",
      "registros": 4,
      "path": "s3://sin-exam-gold-877617909831/kpi_ventas_anual/",
      "metricas": [
        "ventas_totales",
        "ganancia_total",
        "margen_%"
      ]
    },
    {
      "nombre": "KPI Top 20 Productos",
      "registros": 20,
      "path": "s3://sin-exam-gold-877617909831/kpi_top_productos/",
      "metricas": [
        "ventas_totales",
        "ganancia_total",
        "participacion_%"
      ]
    },
    {
      "nombre": "KPI Categor\u00edas",
      "registros": 3,
    

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()